# AML Model Empirical Evaluation

This notebook evaluates the trained model on the strict test split (`dataset_outputs/test.csv`) and reports AML + NER metrics.

In [ ]:
# ============================================
# EMPIRICAL EVALUATION — AML + NER
# ============================================

import pandas as pd
import numpy as np
import pickle
from sklearn.metrics import confusion_matrix, roc_auc_score

# Load data
test_df = pd.read_csv("dataset_outputs/test.csv")

with open("trained_models/forged_document_rf_model.pkl", "rb") as f:
    model = pickle.load(f)

X_test = test_df.drop(columns=["Label", "Image_Name"], errors="ignore")
y_test = test_df["Label"]

expected_features = model.named_steps["preprocessor"].feature_names_in_
for col in expected_features:
    if col not in X_test.columns:
        X_test[col] = 0
X_test = X_test[list(expected_features)]

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:,1]

# ===== AML METRICS =====
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

accuracy = (tp + tn) / (tp + tn + fp + fn)
precision = tp / (tp + fp + 1e-6)
sensitivity = tp / (tp + fn + 1e-6)
specificity = tn / (tn + fp + 1e-6)
fpr = fp / (fp + tn + 1e-6)
roc_auc = roc_auc_score(y_test, y_proba)

print("===== AML METRICS =====")
print(f"Accuracy     : {accuracy:.4f}")
print(f"Precision    : {precision:.4f}")
print(f"Sensitivity  : {sensitivity:.4f}")
print(f"Specificity  : {specificity:.4f}")
print(f"FPR          : {fpr:.4f}")
print(f"ROC-AUC      : {roc_auc:.4f}")
print("Unique predictions:", np.unique(y_pred))

# ===== NER EVALUATION =====

print("\n===== NER EVALUATION =====")

avg_fields = test_df["NER_Field_Count"].mean()
avg_completeness = test_df["Field_Completeness"].mean()

print(f"Avg Fields Extracted   : {avg_fields:.2f}")
print(f"Field Completeness     : {avg_completeness:.4f}")

def ner_recall(row):
    expected = 8
    return row["NER_Field_Count"] / expected

test_df["NER_Recall"] = test_df.apply(ner_recall, axis=1)

print(f"NER Recall (avg): {test_df['NER_Recall'].mean():.4f}")
